# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata_dict = dataset.metadata.to_json()
print(f"{metadata_dict['name']}: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's enumerate all record sets and their fields by `@id` using the dataset metadata. This will guide us in selecting which data to extract.

In [ ]:
# Explore all available record sets and their fields by `@id`
from pprint import pprint
record_sets_info = []

metadata = dataset.metadata
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets():
        rs_id = getattr(rs, '@id', None)
        print(f'Record Set @id: {rs_id}, Name: {getattr(rs, "name", "-") }')
        if hasattr(rs, 'fields'):
            for f in rs.fields():
                f_id = getattr(f, '@id', None)
                print(f'  Field @id: {f_id}, Name: {getattr(f, "name", "-") }, DataType: {getattr(f, "data_type", "-")}')
        print()
        record_sets_info.append(rs_id)
else:
    print("No record sets found in metadata. Attempting to list available record sets directly.")
    # Fallback: attempt to enumerate record sets from JSON
    if 'recordSet' in metadata_dict and isinstance(metadata_dict['recordSet'], list):
        for rs in metadata_dict['recordSet']:
            rs_id = rs.get('@id', None)
            print(f'Record Set @id: {rs_id}')
            record_sets_info.append(rs_id)
    else:
        print("Dataset does not contain explicit recordSet entries.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered in the previous step.

**Note:** For this dataset, the main record set containing the proteomics table is usually called something like `'cr:Proteins'` or similar. We'll demonstrate extracting all available record sets. Adjust the `record_sets` list as needed based on those `@id`s.

In [ ]:
# Obtain all record set @ids that were found (if any), else attempt a default.
record_sets = []

# Repopulate from earlier cell's record_sets_info if present, else fallback
try:
    record_sets = record_sets_info.copy()
except:
    pass
if not record_sets:
    # Fallback: try common record set names
    record_sets = ['cr:Proteins', 'cr:Samples', 'cr:Peptides']

dataframes = {}

for record_set in record_sets:
    print(f"Loading records for Record Set {record_set}")
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"- Extracted {len(df)} records with columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"  Could not extract for {record_set}: {e}\n")

# Show sample columns and rows for the first valid dataframe
if dataframes:
    first_rs = next(iter(dataframes))
    print(f"First record set: {first_rs}")
    print("Columns:", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No dataframes extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes example operations: remove outliers, transform data distributions, or group data for further analysis.

> All columns should be accessed by their `@id` field names.

In [ ]:
import numpy as np
if not dataframes:
    print("No dataframes loaded for EDA.")
else:
    # Use the first dataframe in our dictionary for demo (update as needed by inspection)
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]

    numeric_field = None
    # Try to automatically select a numeric field (float/int) by inspecting first row
    for col in df.columns:
        try:
            if np.issubdtype(df[col].dropna().dtype, np.number):
                numeric_field = col
                break
        except:
            continue
    if not numeric_field:
        print("No obvious numeric field found. Please inspect dataset columns and choose a suitable one by @id.")
    else:
        print(f"Chosen numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 1
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by first non-numeric field
        group_field = None
        for col in df.columns:
            if col != numeric_field and not np.issubdtype(df[col].dropna().dtype, np.number):
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (mean of numeric fields):")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we'll plot the distribution of the selected numeric field.
> Note: Visualization will use the data filtered above if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No data available for plotting.")
elif not numeric_field:
    print("No numeric field available for plotting.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        order = df[group_field].dropna().unique()
        sns.boxplot(x=group_field, y=numeric_field, data=df, order=order)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using `mlcroissant`, examined its structure, extracted key tables via their record set `@id`s, and carried out basic exploratory data analysis and visualization. Further, more domain-specific analyses can be performed once the field meanings are carefully studied.

- For your own research, always refer to record sets, fields, and columns by their canonical `@id` values.
- For deeper domain analysis and protein science insight, consult the detailed Croissant schema and the scientific article accompanying the dataset.